# Merge LoRA Adapter & Push to HuggingFace

This notebook merges `Anushka69/llama3-finetuned-cv-adapter` with its base model
`NousResearch/Meta-Llama-3-8B-Instruct` and pushes the merged model to HuggingFace.

**Requirements:**
- Google Colab with **T4 GPU** (free tier works)
- HuggingFace account with a **Write** access token

**Runtime:** ~10-15 minutes

### Before you start:
1. Go to **Runtime** → **Change runtime type** → Select **T4 GPU** → **Save**
2. Then **Runtime** → **Run all** to execute everything

## Step 1: Install All Dependencies

Installs transformers, peft, bitsandbytes (for 4-bit loading), and torchao.

In [ ]:
!pip install -U transformers accelerate peft safetensors huggingface_hub bitsandbytes>=0.46.1 torchao>=0.16.0
print("\n--- All dependencies installed! ---")

## Step 2: Verify GPU

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_props = torch.cuda.get_device_properties(0)
    vram_gb = gpu_props.total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")
else:
    print("ERROR: No GPU detected! Go to Runtime > Change runtime type > T4 GPU")

## Step 3: Login to HuggingFace

You need a **Write** access token from https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Step 4: Configure Model IDs

Update `MERGED_MODEL_ID` if you want a different name for the merged model.

In [ ]:
BASE_MODEL_ID = "NousResearch/Meta-Llama-3-8B-Instruct"
ADAPTER_ID = "Anushka69/llama3-finetuned-cv-adapter"
MERGED_MODEL_ID = "Anushka69/llama3-cv-merged"

print(f"Base model:    {BASE_MODEL_ID}")
print(f"Adapter:       {ADAPTER_ID}")
print(f"Merged output: {MERGED_MODEL_ID}")

## Step 5: Load Base Model (4-bit quantized to fit T4)

Downloads ~16GB base model and loads it in 4-bit to fit in T4's 15GB VRAM. Takes 3-5 minutes.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

print("Loading base model in 4-bit (to fit in T4 VRAM)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)
print(f"Base model loaded!")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## Step 6: Load & Merge LoRA Adapter

In [ ]:
from peft import PeftModel

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_ID)
print("Adapter loaded!")

print("Merging adapter into base model...")
merged_model = model.merge_and_unload()
print("Merge complete!")

## Step 7: Load Tokenizer

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_ID)
print(f"Tokenizer loaded! Vocab size: {tokenizer.vocab_size:,}")

## Step 8: Quick Test (Optional)

Runs a quick inference to verify the merged model works.

In [ ]:
prompt = "How do I write a strong professional resume?"

messages = [
    {"role": "system", "content": "You are a helpful career advisor specializing in CV and resume writing."},
    {"role": "user", "content": prompt}
]

input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt").to(merged_model.device)

print(f"Prompt: {prompt}\n")
print("Generating response...")

with torch.no_grad():
    output = merged_model.generate(
        input_ids,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.95,
        do_sample=True,
    )

response = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True)
print(f"Response:\n{response}")

## Step 9: Save & Push Merged Model to HuggingFace

Saves the model locally first, then uploads to HuggingFace. Takes 5-10 minutes.

In [ ]:
from huggingface_hub import HfApi

save_dir = "/tmp/merged_model"

print("Saving merged model locally...")
merged_model.save_pretrained(save_dir, safe_serialization=True, max_shard_size="2GB")
tokenizer.save_pretrained(save_dir)
print("Saved locally!")

print(f"\nPushing to {MERGED_MODEL_ID}...")
print("This takes 5-10 minutes. Please wait...\n")

api = HfApi()
api.create_repo(MERGED_MODEL_ID, exist_ok=True)
api.upload_folder(folder_path=save_dir, repo_id=MERGED_MODEL_ID)

print(f"\nDone! Your merged model is at:")
print(f"https://huggingface.co/{MERGED_MODEL_ID}")

## Step 10: Create Model Card

In [ ]:
model_card = f"""---
base_model: {BASE_MODEL_ID}
tags:
  - llama3
  - peft
  - lora
  - merged
  - cv
  - resume
license: llama3
---

# Llama 3 8B - CV/Resume Fine-Tuned (Merged)

This is a merged version of [{ADAPTER_ID}](https://huggingface.co/{ADAPTER_ID})
with [{BASE_MODEL_ID}](https://huggingface.co/{BASE_MODEL_ID}).

## How it was created

- **Base model**: `{BASE_MODEL_ID}`
- **Adapter**: `{ADAPTER_ID}` (LoRA, r=16, alpha=32)
- **Target modules**: q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj
- **Domain**: CV/Resume writing and career advice

## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(\"{MERGED_MODEL_ID}\")
tokenizer = AutoTokenizer.from_pretrained(\"{MERGED_MODEL_ID}\")
```
"""

api.upload_file(
    path_or_fileobj=model_card.encode(),
    path_in_repo="README.md",
    repo_id=MERGED_MODEL_ID,
)
print("Model card uploaded!")

## Done!

Your merged model is now on HuggingFace. Next steps:

1. Go to your model page and verify it looks correct
2. Update your PEFT Platform `backend/.env`:

```
HF_TOKEN=hf_your_token_here
HF_FINETUNED_MODEL_ID=Anushka69/llama3-cv-merged
```

3. Start the backend and frontend — inference will now use the real model!

**Note:** The first inference request may take 1-2 minutes while HuggingFace
loads the model into their serverless infrastructure. Subsequent requests will be faster.